# TechCorp — Fine-tuning Médical LoRA (Mission Expérimentale)

**Rôle : IA | Challenge IA 7h**

Ce notebook fine-tune `microsoft/Phi-3.5-mini-instruct` avec LoRA/QLoRA sur le dataset médical `ruslanmv/ai-medical-chatbot`.

> **Avant de démarrer** : Aller dans `Exécution > Modifier le type d'exécution > GPU T4`


## 1. Installation des dépendances

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.0 trl==0.11.4 \
    bitsandbytes==0.44.1 accelerate==0.34.2 datasets==3.0.1 \
    evaluate sentencepiece einops
print('Installation terminee')

## 2. Vérification GPU

In [ ]:
import torch
print(f'GPU disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Chargement du dataset médical

In [ ]:
from datasets import load_dataset

print('Chargement du dataset ruslanmv/ai-medical-chatbot...')
raw_dataset = load_dataset('ruslanmv/ai-medical-chatbot', split='train')

print(f'Taille totale   : {len(raw_dataset)} exemples')
print(f'Colonnes        : {raw_dataset.column_names}')
print(f'\nExemple :')
print(raw_dataset[0])

## 4. Nettoyage et préparation du dataset

In [ ]:
# Parametres
MAX_SAMPLES    = 5000   # Limite pour tenir dans le temps du hackathon
MIN_ANSWER_LEN = 30     # Caracteres minimum pour une reponse valide
MAX_ANSWER_LEN = 1500   # Caracteres maximum

def format_sample(example):
    """Convertit en format chat Phi-3.5"""
    # Le dataset a les colonnes 'Patient' et 'Doctor'
    patient = (example.get('Patient') or example.get('question') or '').strip()
    doctor  = (example.get('Doctor')  or example.get('answer')   or '').strip()
    
    if not patient or not doctor:
        return None
    if len(doctor) < MIN_ANSWER_LEN or len(doctor) > MAX_ANSWER_LEN:
        return None
    
    return {
        'text': f'<|user|>\n{patient}<|end|>\n<|assistant|>\n{doctor}<|end|>',
        'instruction': patient,
        'output': doctor,
    }

# Nettoyage
cleaned = []
seen    = set()
for ex in raw_dataset:
    formatted = format_sample(ex)
    if formatted is None:
        continue
    key = formatted['instruction'][:100].lower()
    if key in seen:
        continue
    seen.add(key)
    cleaned.append(formatted)
    if len(cleaned) >= MAX_SAMPLES:
        break

print(f'Apres nettoyage : {len(cleaned)} exemples')
print(f'\nExemple format final :')
print(cleaned[0]['text'][:300])

## 5. Conversion en HuggingFace Dataset

In [ ]:
from datasets import Dataset

hf_dataset = Dataset.from_list(cleaned)

# Split train/validation (90/10)
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'Train : {len(train_dataset)} exemples')
print(f'Eval  : {len(eval_dataset)} exemples')

## 6. Chargement du modèle en 4-bit (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

BASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'

# Tokenizer
print('Chargement tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Config 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Chargement modele
print('Chargement modele en 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print('Modele charge en memoire')

## 7. Configuration LoRA

In [ ]:
lora_config = LoraConfig(
    r=8,                   # Rang (meme config que le modele financier heritage)
    lora_alpha=16,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Tokenization du dataset

In [ ]:
MAX_LENGTH = 512

def tokenize(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

print('Tokenization...')
train_tok = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset.column_names)
eval_tok  = eval_dataset.map(tokenize,  batched=True, remove_columns=eval_dataset.column_names)
print(f'Train tokenise : {len(train_tok)} exemples')
print(f'Eval tokenise  : {len(eval_tok)} exemples')

## 9. Entraînement

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir='./medical_lora_output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    remove_unused_columns=False,
    dataloader_drop_last=True,
    report_to='none',
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
)

print('Debut de l entrainement...')
train_result = trainer.train()
print(f'\nEntrainement termine !')
print(f'Loss finale : {train_result.training_loss:.4f}')

## 10. Métriques et évaluation

In [ ]:
import math

# Perplexite sur le set de validation
eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results['eval_loss'])

print('=== METRIQUES FINALES ===')
print(f"Loss d'eval      : {eval_results['eval_loss']:.4f}")
print(f"Perplexite       : {perplexity:.2f}")
print(f"Loss d'entrainement : {train_result.training_loss:.4f}")
print(f"Steps total      : {train_result.global_step}")

## 11. Test du modèle fine-tuné

In [ ]:
model.eval()

test_questions = [
    'What are the common symptoms of diabetes?',
    'I have a headache and fever since yesterday. What could it be?',
    'What is the difference between a virus and a bacteria?',
    'Can anxiety cause physical symptoms?',
]

def generate(prompt, max_new_tokens=150):
    formatted = f'<|user|>\n{prompt}<|end|>\n<|assistant|>\n'
    inputs    = tokenizer(formatted, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.4,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('=== TESTS DU MODELE MEDICAL ===')
for q in test_questions:
    print(f'\nQ: {q}')
    print(f'R: {generate(q)}')
    print('-' * 50)

## 12. Sauvegarde de l'adapter LoRA

In [ ]:
OUTPUT_DIR = './medical_lora_adapter'

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'Adapter LoRA sauvegarde dans : {OUTPUT_DIR}')

import os
files = os.listdir(OUTPUT_DIR)
print(f'Fichiers produits : {files}')

## Résumé pour le rapport

Copie les valeurs ci-dessous dans ton rapport IA :

| Paramètre | Valeur |
|---|---|
| Modèle de base | `microsoft/Phi-3.5-mini-instruct` |
| Dataset | `ruslanmv/ai-medical-chatbot` |
| Exemples utilisés | 5 000 |
| Méthode | QLoRA 4-bit, r=8, alpha=16 |
| Epochs | 3 |
| Loss finale | *(voir cellule 10)* |
| Perplexité | *(voir cellule 10)* |

> **Note** : Ce modèle est expérimental, pas destiné à la production médicale réelle.